# 1.1 FTP-Server data access

## Download raw files and transform them in readable CSV

In [1]:
# filename = 'ADTC_8_adz_raw_20260308'
# filename = 'D_05_adz_raw_20260301'
# filename = 'ADTC1_adz_raw_20260615'
filename = 'ADTC_83_2_L_adz_raw_20260812'

# image_data = 'combined' # path where the images with 2 channels are saved
image_data = 'combined'

In [2]:
from FTP.data_download import downloader
from FTP.data_decoder import decoder

downloader(filename)
decoder(filename)

Connection with geo-amberg.ch
Successful access to /45-M-02075 SG Notsanierung/ADTC/data/ADTC_RAW
All files successfully downloadeed
All files successfully decoded


## DataFrame for the CSV (filename, timestamp)

In [3]:
import os
import pandas as pd
from pathlib import Path
from utils import date_extractor

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

path = root / f'RAW_DATA/{filename}'


rows = []
for file in os.listdir(path):
    file_path = os.path.join(path, file)

    with open(file_path, 'r') as f:
        content = f.read()
        micro_sec = int(content.split(';')[0][4:]) # in the raw data -> get the timestamp in ms

    time = date_extractor(micro_sec) # function defined in utils.py -> return the time in the day !

    date = os.path.basename(file_path).split('_')[-2]
    yyyy = date[:4]
    mm = date[4:6]
    dd = date[6:]

    timestamp = f'{yyyy}-{mm}-{dd} {time}' # timestamp in format: YYYY-MM-DD HH:MM:SS
    timestamp = pd.to_datetime(timestamp)

    rows.append({
        'file': file.split('.')[0],
        'timestamp_raw': timestamp
    })

df_raw_data = pd.DataFrame(rows)
df_raw_data.head(5)

""


# 1.2 GeoVIS-API data access

## Get sensor data from the GeoVis

In [93]:
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
import pandas as pd
from utils import main

env_path = 'GeoVis/DTC.env'
load_dotenv(env_path)

LOGIN = os.getenv('GEOVIS_LOGIN')
PASSWORD = os.getenv('GEOVIS_PASSWORD')


# project ID (change it for each project)
if filename.startswith('ADTC_'): # Ruschlikon
    PROJECT_ID = '1113' # Ruschlikon
elif filename.startswith('D'): # Opfkon
    PROJECT_ID = '817' # Opfikon
elif filename.startswith('ADTC1'): #Rastatt
    PROJECT_ID = '1191' # Rastatt
else:
    print('Error with the filename')

login_data = {'Login': LOGIN, 'Password': PASSWORD}
sensor_name = '_Peak'

if PROJECT_ID == "1113":
    Projekt_DB = f"{PROJECT_ID}_0"
elif PROJECT_ID == "817":
    Projekt_DB = f"{PROJECT_ID}_7"
elif PROJECT_ID == "1191":
    Projekt_DB = f"{PROJECT_ID}_0"
else:
    raise ValueError("Unknown PROJECT_ID")

datum = filename.split('_')[-1]

dfs_raw_3 = main(login_data, PROJECT_ID, Projekt_DB, sensor_name, datum)
print('Total number of data on GeoVIS for this date:', len(dfs_raw_3))

Start Calculation: 2026-06-15T00:00:00
End Calculation  : 2026-06-16T00:00:00
Datenbezug abgeschlossen.
Gefundene Sensoren: 12
Peak Detektion abgeschlossen.
Lokale Paar- und Sensorgeschwindigkeiten gespeichert.
Total number of data on GeoVIS for this date: 150


### Train passage over sensor

In [94]:
from GeoVis.Preprocessing.Peak_Detektion import detect_peaks

dfs_raw = detect_peaks(dfs_raw_3)

all_peaks = []
result = []

for val in dfs_raw.values():
    peaks = val["Peaks"]

    if filename.startswith('ADTC_'): # Rueschlikon
        peaks_sensor = peaks[peaks["Sensor"].str.contains("ADTC_8")] # Rueschlikon (sensor ADTC_8)
    elif filename.startswith('D'): # Opfikon
        peaks_sensor = peaks[peaks["Sensor"].str.contains("D_05")] # Opfikon (sensor D_05)
    elif filename.startswith('ADTC'): # Rastatt
            peaks_sensor = peaks[peaks["Sensor"].str.contains("ADTC1_")] # Rastatt (sensor ADTC1)

    all_peaks.append(peaks_sensor[["Time", "Value", "Sensor"]])

peaks_df = pd.concat(all_peaks, ignore_index=True)

peaks_df["Time"] = pd.to_datetime(peaks_df["Time"])
peaks_df = peaks_df.sort_values('Time').reset_index(drop=True)

time_diff = peaks_df["Time"].diff().dt.total_seconds().fillna(0)

peaks_df["train_nr"] = (time_diff>60).cumsum()+1

for train_nr, group in peaks_df.groupby("train_nr"):
    times = group["Time"].tolist()
    
    timestamp = times[0]

    train_deltas = [
        (times[i] - times[i-1]).total_seconds()
        for i in range(1, len(times))
    ]

    result.append({
        "timestamp": timestamp,
        "timestamp_serie": times,
        "delta_t": train_deltas
    })

df_geovis_time = pd.DataFrame(result)
df_geovis_time.head(5)

Peak Detektion abgeschlossen.


,timestamp,timestamp_serie,delta_t
0,2026-06-15 00:27:42.139,"[2026-06-15 00:27:42.139000, 2026-06-15 00:27:...","[0.12, 0.76, 0.12, 0.21, 0.12, 0.785, 0.117, 0..."
1,2026-06-15 00:32:56.261,"[2026-06-15 00:32:56.261000, 2026-06-15 00:32:...","[0.121, 0.37, 0.122, 0.221, 0.084, 0.344, 0.08..."
2,2026-06-15 00:40:00.715,"[2026-06-15 00:40:00.715000, 2026-06-15 00:40:...","[0.124, 0.37, 0.124]"
3,2026-06-15 00:54:16.755,"[2026-06-15 00:54:16.755000, 2026-06-15 00:54:...","[0.105, 0.583, 0.12, 0.616, 0.12, 0.583, 0.108]"
4,2026-06-15 01:01:42.948,"[2026-06-15 01:01:42.948000, 2026-06-15 01:01:...","[0.146, 0.328, 0.144, 0.305, 0.122, 0.378, 0.118]"


### Velocity over sensor

In [95]:
# use dfs_raw_3 -> results of the function 'main()'

rows = []

for ts, data in dfs_raw_3.items():
    sensor = data.get('geschw_pro_achse_sensor_local')

    if sensor is None:
        continue

    # replace 5 with the sensor you want to use to extract data
    s_id = 1
    if s_id in sensor:
        rows.append({
            'timestamp': ts,
            'velocity': sensor[s_id]
        })

df_geovis_velocity = pd.DataFrame(rows)
df_geovis_velocity.head(5)

,timestamp,velocity
0,2026-06-15 00:27:32.170,"[21.374729691864434, 21.097272270334212, 21.23..."
1,2026-06-15 00:32:46.249,"[20.83643307047563, 21.136194492681717, 20.978..."
2,2026-06-15 00:39:50.699,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
3,2026-06-15 00:54:06.721,"[22.261426993831982, 22.226805702037897, 21.67..."
4,2026-06-15 01:01:32.259,"[21.088180814638804, 20.90724385451653, 21.513..."


### Merge both GeoVis DataFrame and compute traing length

In [96]:
df_geovis = pd.merge_asof(
    df_geovis_time.sort_values('timestamp'),
    df_geovis_velocity.sort_values('timestamp'),
    left_on='timestamp',
    right_on='timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('20s')
)

# add the length between every axis, computed using d=v*t
df_geovis['length'] = df_geovis.apply(
    lambda row: [
        a * b for a, b in zip(row["delta_t"], row["velocity"])
    ] if isinstance(row["delta_t"], list)
    and isinstance(row["velocity"], list)
    and len(row["delta_t"]) > 1
    and len(row["velocity"]) > 1
    else [],
    axis=1
)

# add the total train length, by summin the values in the length list
df_geovis['total_length'] = df_geovis['length'].apply(sum)
df_geovis = df_geovis[df_geovis['total_length'] != 0] # remove all entries where the total length is 0

df_geovis.head(5)
print(len(df_geovis))

126


## Merge DataFrame from the CSV and DataFrame from GeoVIS

In [97]:
df_merged = pd.merge_asof(
    df_geovis.sort_values('timestamp'),
    df_raw_data.sort_values('timestamp_raw'),
    left_on='timestamp',
    right_on='timestamp_raw',
    direction='nearest',
    tolerance=pd.Timedelta('20s')
)

df_merged = df_merged.dropna(subset=['file', 'timestamp_raw'])

cols = ['file', 'timestamp_raw'] + [
    c for c in df_merged.columns
    if c not in ['file', 'timestamp_raw']
]

df_merged = df_merged[cols]

df_merged.head(5)
print(len(df_merged))

126


# 1.3 Combine FTP & API data

## Cut the CSV (First-Last Peak)

In [98]:
import numpy as np
from scipy.signal import savgol_filter
from peaks import peakFind, firstLastPeak

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

src = root / f'CSV_DATA/{filename}_csv'
dst = root / f'CSV_DATA/{filename}_cut'
os.makedirs(dst, exist_ok=True)

for file in os.listdir(src):
    file_path = os.path.join(src, file)
    df = pd.read_csv(file_path)

    time = df['Time [s]'].values
    signal = df['Distance[mm]'].values

    fs = 1/np.mean(np.diff(time))
    signal_smooth = savgol_filter(signal, 101, 3)

    # peak detection
    peakValues, peakTimes = peakFind(time, signal_smooth, fs)
    firstValue, firstTime, lastValue, lastTime = firstLastPeak(peakValues, peakTimes)

    name = file.split('.')[0]

    # add first peak in df_merged
    # df_merged['first_peak_raw'] = firstTime
    df_merged.loc[df_merged['file'] == name, 'first_peak_raw'] = firstTime

    # cut the csv
    subset = df_merged.loc[df_merged['file'] == name, 'velocity']
    if subset.empty:
        print(f'Skipping {file}')
        continue

    v = subset.iloc[0]
    velocity_start = v[0] if isinstance(v, list) else v
    velocity_end = v[-1] if isinstance(v, list) else v

    start_sec_m = 1/velocity_start
    end_sec_m = 1/velocity_end

    start = max(0, firstTime - start_sec_m)
    end = lastTime + end_sec_m
    mask = (df["Time [s]"] >= start) & (df["Time [s]"] <= end)
    df_cut = df.loc[mask].copy()

    if df_cut.empty:
        print(f"Empty cut for file: {file}")
        continue

    out_path = os.path.join(dst, file.replace(".csv", "_cut.csv"))
    df_cut.to_csv(out_path, index=False)

df_merged.head(5)

Empty cut for file: ADTC1_adz_raw_20260615_0041.csv
Skipping ADTC1_adz_raw_20260615_0254.csv
Skipping ADTC1_adz_raw_20260615_0427.csv
Empty cut for file: ADTC1_adz_raw_20260615_0431.csv
Skipping ADTC1_adz_raw_20260615_0438.csv
Skipping ADTC1_adz_raw_20260615_0442.csv
Skipping ADTC1_adz_raw_20260615_0503.csv
Skipping ADTC1_adz_raw_20260615_0508.csv
Skipping ADTC1_adz_raw_20260615_0518.csv
Empty cut for file: ADTC1_adz_raw_20260615_0535.csv
Skipping ADTC1_adz_raw_20260615_0554.csv
Skipping ADTC1_adz_raw_20260615_0603.csv
Skipping ADTC1_adz_raw_20260615_0622.csv
Empty cut for file: ADTC1_adz_raw_20260615_0638.csv
Skipping ADTC1_adz_raw_20260615_0654.csv
Skipping ADTC1_adz_raw_20260615_0803.csv
Skipping ADTC1_adz_raw_20260615_0903.csv
Skipping ADTC1_adz_raw_20260615_0912.csv
Empty cut for file: ADTC1_adz_raw_20260615_0917.csv
Skipping ADTC1_adz_raw_20260615_1049.csv
Skipping ADTC1_adz_raw_20260615_1205.csv
Skipping ADTC1_adz_raw_20260615_1246.csv
Skipping ADTC1_adz_raw_20260615_1307.csv
Em

,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length,first_peak_raw
0,ADTC1_adz_raw_20260615_0028,2026-06-15 00:27:32.099,2026-06-15 00:27:42.139,"[2026-06-15 00:27:42.139000, 2026-06-15 00:27:...","[0.12, 0.76, 0.12, 0.21, 0.12, 0.785, 0.117, 0...","[21.374729691864434, 21.097272270334212, 21.23...","[2.564967563023732, 16.033926925454, 2.5476062...",99.078036,9.922365
1,ADTC1_adz_raw_20260615_0034,2026-06-15 00:32:46.234,2026-06-15 00:32:56.261,"[2026-06-15 00:32:56.261000, 2026-06-15 00:32:...","[0.121, 0.37, 0.122, 0.221, 0.084, 0.344, 0.08...","[20.83643307047563, 21.136194492681717, 20.978...","[2.521208401527551, 7.820391962292235, 2.55937...",2925.141415,9.917025
2,ADTC1_adz_raw_20260615_0041,2026-06-15 00:39:50.668,2026-06-15 00:40:00.715,"[2026-06-15 00:40:00.715000, 2026-06-15 00:40:...","[0.124, 0.37, 0.124]","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan]",NaN,9.936346
3,ADTC1_adz_raw_20260615_0055,2026-06-15 00:54:06.714,2026-06-15 00:54:16.755,"[2026-06-15 00:54:16.755000, 2026-06-15 00:54:...","[0.105, 0.583, 0.12, 0.616, 0.12, 0.583, 0.108]","[22.261426993831982, 22.226805702037897, 21.67...","[2.337449834352358, 12.958227724288093, 2.6006...",49.536474,9.931430
4,ADTC1_adz_raw_20260615_0102,2026-06-15 01:01:32.921,2026-06-15 01:01:42.948,"[2026-06-15 01:01:42.948000, 2026-06-15 01:01:...","[0.146, 0.328, 0.144, 0.305, 0.122, 0.378, 0.118]","[21.088180814638804, 20.90724385451653, 21.513...","[3.0788743989372653, 6.857575984281422, 3.0980...",32.291124,9.912503


## Add length to the CSV

In [99]:
for file in os.listdir(dst):

    file_path = os.path.join(dst, file)
    df_cut = pd.read_csv(file_path)

    name = file.split("_cut")[0]

    # get merged row
    row = df_merged.loc[df_merged["file"] == name]

    if row.empty:
        print(f"Missing in df_merged: {name}")
        continue

    row = row.iloc[0]


    times = pd.to_datetime(row["timestamp_serie"])
    peak_times = pd.to_datetime(times).values.astype("datetime64[ns]")

    t0 = peak_times[0]
    t_peaks = (peak_times - t0) / np.timedelta64(1, "s")

    cum_length = np.concatenate([[0], np.cumsum(row["length"])])


    # first peak reference (from cut file)
    time_shift = df_cut["Time [s]"].values
    time_shift = time_shift - time_shift[0]  # ensure starts at 0

    # interpolate length
    min_len = min(len(t_peaks), len(cum_length))

    t_peaks = t_peaks[:min_len]
    cum_length = cum_length[:min_len]

    df_cut["Length [m]"] = np.interp(
        time_shift,
        t_peaks,
        cum_length
    )

    # save file
    df_cut.to_csv(file_path, index=False)

In [100]:
nb_csv = len([f for f in os.listdir(dst) if f.endswith('.csv')])
print(f"Total CSV: {nb_csv}")

Total CSV: 113


In [101]:
df_merged.to_csv(f'DataFrame/df_merged_{filename}.csv', index=False)

## Extend CSV to 400 m

In [102]:
import os
import numpy as np
import pandas as pd

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

input_path = root / f'CSV_DATA/{filename}_cut'
output_path = root / f'CSV_DATA/{filename}_ext'
os.makedirs(output_path, exist_ok=True)

target_length = 400

for file in os.listdir(input_path):
    file_path = os.path.join(input_path, file)
    df = pd.read_csv(file_path)

    basename = file.split('_cut')[0]

    if not basename in df_merged['file'].values:
        continue
    merged_row = df_merged[df_merged['file'] == basename]
    if merged_row.empty:
        continue

    # load data
    time = df['Time [s]'].values
    length = df['Length [m]'].values
    signal = df['Distance[mm]'].values

    mask = np.isfinite(time) & np.isfinite(length) & np.isfinite(signal)
    time = time[mask]
    length = length[mask]
    signal = signal[mask]

    if len(length) < 5:
        continue

    # original time = 0
    time = time-time[0]
    length = length - length[0]

    dl = np.diff(length)
    dt = np.diff(time)

    valid = (dl > 0) & np.isfinite(dl) & np.isfinite(dt)

    if np.sum(valid) < 2:
        continue

    time_per_meter = np.mean(dt[valid] / dl[valid])

    if not np.isfinite(time_per_meter) or time_per_meter <= 0:
        continue

    # check end length
    if length[-1] > target_length:
        mask = length <= target_length
        time = time[mask]
        length = length[mask]
        signal = signal[mask]

    last_time = time[-1]
    last_length = length[-1]
    # last_time = time[-1]
    # last_length = length[-1]

    # if last_length > target_length:
    #     continue

    # extension
    step_length = np.mean(dl[valid])

    if not np.isfinite(step_length) or step_length <= 0:
        continue

    missing_length = np.arange(
        last_length + step_length,
        target_length + step_length,
        step_length
    )

    extra_time = last_time + (missing_length - last_length) * time_per_meter

    # safety check
    if np.any(~np.isfinite(extra_time)):
        continue

    extra_signal = np.zeros_like(missing_length)

    # concatenate
    time_ext = np.concatenate([time, extra_time])
    length_ext = np.concatenate([length, missing_length])
    signal_ext = np.concatenate([signal, extra_signal])

    # force 400 m endpoint
    time_ext = np.append(time_ext, extra_time[-1] + time_per_meter)
    length_ext = np.append(length_ext, target_length)
    signal_ext = np.append(signal_ext, 0)

    # cleaning
    mask_final = np.isfinite(time_ext) & np.isfinite(length_ext) & np.isfinite(signal_ext)

    time_ext = time_ext[mask_final]
    length_ext = length_ext[mask_final]
    signal_ext = signal_ext[mask_final]

    # save new extended CSV
    df_out = pd.DataFrame({
        "Time [s]": time_ext,
        "Length [m]": length_ext,
        "Distance[mm]": signal_ext
    })

    out_filename = basename + "_ext.csv"

    df_out.to_csv(os.path.join(output_path, out_filename), index=False)

# 1.4: CWT Image generation

## Generate CWT images

In [103]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pywt
from PIL import Image

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

path = root / f'CSV_DATA/{filename}_ext'

target_length = 400
n_length = 200

for file in os.listdir(path):
    file_path = os.path.join(path, file)
    df = pd.read_csv(file_path)

    file_name = file.split('_ext')[0]
    match = df_merged[df_merged['file'] == file_name]

    if match.empty:
        print(f'No match for {file_name}')
        continue

    time = df['Time [s]'].values
    length = df['Length [m]'].values
    signal = df['Distance[mm]'].values

    # # shift to zero / clean the data to be sure
    # time = time - time[0]
    # length = length - length[0]
    # mask = np.isfinite(time) & np.isfinite(signal) & np.isfinite(length)

    # time = time[mask]
    # length = length[mask]
    # signal = signal[mask]

    # Downsampling - to speed up CWT
    decim = 5
    time_ds = time[::decim]
    length_ds = length[::decim]
    signal_ds = signal[::decim]

    # -------------------------------------------
    #    #######        ##      ##      ########
    #   ##              ##      ##         ##  
    #   ##              ##      ##         ##  
    #   ##              ##  ##  ##         ##
    #   ##              ##  ##  ##         ##
    #    #######         ###  ###          ## 
    # ------------------------------------------

    # Continuous Wavelet Transform
    dt = np.mean(np.diff(time_ds))                  # time step
    if not np.isfinite(dt) or dt <= 0:
        continue
    fs = 1 / dt                                     # frequency
    freqs = np.linspace(0.2, 5, 100)                # Define frequency range
    cf = pywt.central_frequency('cmor1.5-1.0')      # CWT using Morlet wavelet
    scales = cf * fs / freqs                        # convert frequency to a scale
    scales = np.clip(scales, 1, 1000)
    coeffs, _ = pywt.cwt(                           # Compute CWT
        signal_ds,
        scales,
        'cmor1.5-1.0',
        sampling_period=dt
    )
    power = np.abs(coeffs) ** 2                     # power spectrum

    # Normalize length
    length_ds = length_ds - length_ds.min()
    if length_ds.max() <= 0:
        continue
    length_ds = length_ds / length_ds.max() * target_length

    # fixed grid
    x_out = np.linspace(0, target_length, n_length) # images with 200 pixels width
    power_resampled = np.zeros((power.shape[0], n_length))
    max_len = length_ds[-1]
    max_idx = np.searchsorted(x_out, max_len)
    if max_idx < 2:
        continue

    # interpolation
    for i in range(power.shape[0]):
        power_resampled[i, :max_idx] = np.interp( # resample wavelet result
            x_out[:max_idx],
            length_ds,
            power[i, :]
        )

    # generate plots (images)
    fig, ax1 = plt.subplots(figsize=(5.12, 5.12), dpi=100)
    fig.patch.set_facecolor('white')
    ax1.set_facecolor('white')
    ax1.imshow(
        power_resampled,
        extent=[0, target_length, freqs[0], freqs[-1]],
        aspect='auto',
        cmap='Greys_r',
        origin='lower'
    )
    ax1.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    # save
    out_path = root / f'images/total_length'
    os.makedirs(out_path, exist_ok=True)
    out_filename = os.path.splitext(os.path.basename(file_path))[0] + ".png"
    img_path = os.path.join(out_path, out_filename)
    fig.savefig(
        img_path,
        dpi=100,
        bbox_inches='tight',
        pad_inches=0,
        facecolor='white'
    )

    # remove alpha channel (RGBA -> RGB)
    Image.open(img_path).convert('RGB').save(img_path)
    plt.close(fig)

## Detect 4 Peaks -> CWT

In [104]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

from peaks import peakFind, firstLastPeak, downPeak

path = root / f'CSV_DATA/{filename}_ext'
out_dir = root / f'CSV_DATA/{filename}_ext_lok'
os.makedirs(out_dir, exist_ok=True)

for data in os.listdir(path):
    data = os.path.join(path, data)
    df = pd.read_csv(data)

    time = df['Time [s]'].values
    signal = df['Distance[mm]'].values

    fs = 1 / np.mean(np.diff(time))
    signal_smooth = savgol_filter(signal, 101, 3)

    peakValues, peakTimes = peakFind(time, signal_smooth, fs)
    firstValue, firstTime, lastValue, lastTime = firstLastPeak(peakValues, peakTimes)
    downValues, downTimes = downPeak(peakValues, peakTimes)

    # define the end time:
    if len(downTimes) >= 5: # if there are more than 5 downTimes take the middle of 4 and 5
        end = (downTimes[3] + downTimes[4]) / 2

    elif len(downTimes) == 4: # if there are only 4 take the last time with a value
        # take the last non-zero signal value
        valid_idx = np.where(signal != 0)[0]
        if len(valid_idx) == 0:
            continue
        
        end = time[valid_idx[-1]]

    else: # if there are less than 4, the simply ignor this file !
        continue
            
    start = firstTime

    df_cut = df[(df['Time [s]'] >= start) & (df['Time [s]'] <= end)].copy()

    file = os.path.splitext(os.path.basename(data))[0]
    out_path = os.path.join(out_dir, f'{file}_lok.csv')
    df_cut.to_csv(out_path, index=False)

In [105]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pywt
from PIL import Image

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

path = root / f'CSV_DATA/{filename}_ext_lok'

target_length = 50 # to ensure that all lok length can be represented !
n_length = 200

for file in os.listdir(path):
    file_path = os.path.join(path, file)
    df = pd.read_csv(file_path)

    file_name = file.split('_ext')[0]
    match = df_merged[df_merged['file'] == file_name]

    if match.empty:
        print(f'No match for {file_name}')
        continue

    time = df['Time [s]'].values
    length = df['Length [m]'].values
    signal = df['Distance[mm]'].values

    # # shift to zero / clean the data to be sure
    # time = time - time[0]
    # length = length - length[0]
    # mask = np.isfinite(time) & np.isfinite(signal) & np.isfinite(length)

    # time = time[mask]
    # length = length[mask]
    # signal = signal[mask]

    # Downsampling - to speed up CWT
    decim = 1 # because it's much shorter ! (before we used 5)
    time_ds = time[::decim]
    length_ds = length[::decim]
    signal_ds = signal[::decim]

    # -------------------------------------------
    #    #######        ##      ##      ########
    #   ##              ##      ##         ##  
    #   ##              ##      ##         ##  
    #   ##              ##  ##  ##         ##
    #   ##              ##  ##  ##         ##
    #    #######         ###  ###          ## 
    # ------------------------------------------

    # Continuous Wavelet Transform
    dt = np.mean(np.diff(time_ds))                  # time step
    if not np.isfinite(dt) or dt <= 0:
        continue
    fs = 1 / dt                                     # frequency
    freqs = np.linspace(0.2, 15, 100)                # Define frequency range
    cf = pywt.central_frequency('cmor1.5-1.0')      # CWT using Morlet wavelet
    scales = cf * fs / freqs                        # convert frequency to a scale
    scales = np.clip(scales, 1, 1000)
    coeffs, _ = pywt.cwt(                           # Compute CWT
        signal_ds,
        scales,
        'cmor1.5-1.0',
        sampling_period=dt
    )
    power = np.abs(coeffs) ** 2                     # power spectrum

    # Normalize length
    length_ds = length_ds - length_ds.min()
    signal_length = length_ds.max()

    if signal_length > target_length:
        continue
    
    padding = (target_length - signal_length) / 2
    length_ds = length_ds + padding

    x_out = np.linspace(0, target_length, n_length)
    power_resampled = np.zeros((power.shape[0], n_length))

    for i in range(power.shape[0]):
        power_resampled[i, :] = np.interp(
            x_out,
            length_ds,
            power[i, :],
            left=0,
            right=0
        )

    # generate plots (images)
    fig, ax1 = plt.subplots(figsize=(5.12, 5.12), dpi=100)
    fig.patch.set_facecolor('white')
    ax1.set_facecolor('white')
    ax1.imshow(
        power_resampled,
        extent=[0, target_length, freqs[0], freqs[-1]],
        aspect='auto',
        cmap='Greys_r',
        origin='lower'
    )
    ax1.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    # save
    out_path = root / f'images/lok'
    os.makedirs(out_path, exist_ok=True)
    out_filename = os.path.splitext(os.path.basename(file_path))[0] + ".png"
    img_path = os.path.join(out_path, out_filename)
    fig.savefig(
        img_path,
        dpi=100,
        bbox_inches='tight',
        pad_inches=0,
        facecolor='white'
    )

    # remove alpha channel (RGBA -> RGB)
    Image.open(img_path).convert('RGB').save(img_path)
    plt.close(fig)

## Combine train (400 m) and lok together

In [106]:
import os
import numpy as np
from PIL import Image

# Folders
full_dir = 'C:/git/RailwAI/images/total_length'
lok_dir = 'C:/git/RailwAI/images/lok'
out_dir = f'C:/git/RailwAI/images/{image_data}'

os.makedirs(out_dir, exist_ok=True)

for file in os.listdir(full_dir):

    # Full train image
    full_path = os.path.join(full_dir, file)

    # Corresponding locomotive image
    name, ext = os.path.splitext(file)

    lok_file = f"{name}_lok{ext}"
    out_file = f"{name.replace('_ext', '')}{ext}"

    lok_path = os.path.join(lok_dir, lok_file)
    if not os.path.exists(lok_path):
        print(f"Missing locomotive image: {file}")
        continue

    # Read images
    full_img = np.array(Image.open(full_path).convert("RGB"))
    lok_img = np.array(Image.open(lok_path).convert("RGB"))

    # Take only one channel (all 3 are identical)
    full_channel = full_img[:, :, 0]
    lok_channel = lok_img[:, :, 0]

    # Ensure same size
    if full_channel.shape != lok_channel.shape:
        print(f"Shape mismatch for {file}")
        continue

    # Empty third channel
    empty_channel = np.zeros_like(full_channel)

    # Stack into RGB image
    combined = np.stack(
        [full_channel, lok_channel, empty_channel],
        axis=2
    ).astype(np.uint8)

    # Save
    Image.fromarray(combined).save(os.path.join(out_dir, out_file))

Missing locomotive image: ADTC1_adz_raw_20260611_2020_ext.png
Missing locomotive image: ADTC1_adz_raw_20260614_1205_ext.png
Missing locomotive image: ADTC1_adz_raw_20260615_0541_ext.png
Missing locomotive image: ADTC1_adz_raw_20260615_1330_ext.png
Missing locomotive image: ADTC_8_adz_raw_20260305_1340_ext.png
